
# C02a — StyleRemix paper reproduction (AuthorMix-Blog)

Goal: reproduce the released StyleRemix **adapter-merging (AM)** result on AuthorMix-Blog before using StyleRemix as a RELIC intervention.

Paper target (Table 2 / Table 6, Blog, AM, best \(k=3\)):

| metric | paper |
|---|---:|
| Drop rate | 41.0% |
| Grammar | 64.9% |
| Content similarity | 73.7% |
| Overall product | 19.6% |

The released `obfuscate.py` has several execution typos, so this notebook reconstructs the published pipeline from released assets/helpers instead of invoking that script wholesale.

**8 GB hardware deviation:** the generation cell uses Llama-3 8B in 4-bit NF4 by default. This is a low-resource reproduction of the same base checkpoint + released LoRAs, not a bit-identical precision reproduction.


In [1]:
# %pip install peft bitsandbytes textstat Unidecode

In [2]:
# %pip install nevergrad

In [3]:
from huggingface_hub import login

login()

/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained(
    "meta-llama/Meta-Llama-3-8B"
)

print(cfg.model_type)
print("PASS: Llama-3-8B access works.")

llama
PASS: Llama-3-8B access works.


In [5]:

from pathlib import Path
import os, sys, json, time, gc, subprocess, importlib.util
import numpy as np, pandas as pd, torch

ROOT = Path("..")
SR_DIR = ROOT / "external" / "StyleRemix"
OUT = ROOT / "results" / "causal" / "C02a_styleremix_reproduction"
OUT.mkdir(parents=True, exist_ok=True)
SR_DIR.parent.mkdir(parents=True, exist_ok=True)

SEED = 42
TOP_K = 3
USE_4BIT = True
N_PER_AUTHOR = None   # set 3 for a smoke test

np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))


CUDA: True
NVIDIA RTX 2000 Ada Generation Laptop GPU


In [6]:

REPO_URL = "https://github.com/jfisher52/StyleRemix.git"
if not SR_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(SR_DIR)], check=True)

commit = subprocess.check_output(
    ["git", "-C", str(SR_DIR), "rev-parse", "HEAD"],
    text=True
).strip()

sys.path.insert(0, str(SR_DIR))
print("StyleRemix commit:", commit)

required = {
    "transformers":"transformers", "peft":"peft", "bitsandbytes":"bitsandbytes",
    "accelerate":"accelerate", "sentence_transformers":"sentence-transformers",
    "textstat":"textstat", "unidecode":"Unidecode", "nltk":"nltk"
}
missing = [p for m,p in required.items() if importlib.util.find_spec(m) is None]
print("missing:", missing)
if missing:
    print("\nRun this in a dedicated StyleRemix kernel, then restart it:")
    print("%pip install " + " ".join(missing))


StyleRemix commit: c999002c4a5e858ac452794b20b896d06137a13c
missing: []


In [7]:
import os
from pathlib import Path
import nltk

NLTK_DIR = Path.home() / "nltk_data"
NLTK_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

os.environ["NLTK_DATA"] = str(
    NLTK_DIR
)

if str(NLTK_DIR) not in nltk.data.path:
    nltk.data.path.insert(
        0,
        str(NLTK_DIR),
    )

for pkg in [
    "punkt",
    "averaged_perceptron_tagger",
]:
    nltk.download(
        pkg,
        download_dir=str(NLTK_DIR),
        quiet=True,
    )

print(
    "NLTK data:",
    NLTK_DIR,
)

from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification,
    BitsAndBytesConfig, pipeline
)
from peft import PeftModel
from sentence_transformers import SentenceTransformer

BLOG_JSON = SR_DIR / "data/test_data/AuthorMix_blog.json"
AVG_PATH = SR_DIR / "data/test_data/StyleMix_averages_by_author"

with open(BLOG_JSON, encoding="utf-8") as f:
    blog_data = json.load(f)

if N_PER_AUTHOR is not None:
    blog_data = {a: xs[:N_PER_AUTHOR] for a,xs in blog_data.items()}

avg = pd.DataFrame(torch.load(AVG_PATH, map_location="cpu", weights_only=False))

print({a:len(x) for a,x in blog_data.items()})
print("total:", sum(map(len, blog_data.values())))
print("average matrix:", avg.shape)


/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/user/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):


NLTK data: /home/user/nltk_data
{'blog_5_1': 200, 'blog_5_2': 200, 'blog_5_3': 200, 'blog_5_4': 200, 'blog_5_5': 200}
total: 1000
average matrix: (26, 14)



## Reproduce automatic style-axis selection

StyleRemix represents each author by measured style coordinates, selects the coordinates farthest from the same-domain mean, and steers the selected axes in the opposite direction. Blog's reported AM optimum uses the top **3** axes.


In [8]:
# ============================================================
# StyleRemix automatic style-axis selection
#
# Faithful compatibility implementation of the released
# identitify_styles_to_change_average().
#
# Only change from released code:
#   Series[index]  ->  Series.iloc[index]
#
# Also uses the paper's AM weight rule:
#   "threshold" = {0.7, 0.9, 1.2, 1.5}
# ============================================================

import random

from src.utils import (
    group_by_classifier,
    normalize_row,
    choose_weights,
)


def identify_styles_to_change_average_compat(
    df,
    top_n,
    magnitude_selection="threshold",
    random_styles=False,
):

    styles_to_change = {}

    for author in df.columns:

        # Same-domain authors only.
        classifier_specific_df = (
            group_by_classifier(
                df,
                author,
            )
        )

        # Exact released normalization.
        normalized_df = (
            classifier_specific_df.apply(
                normalize_row,
                axis=1,
            )
        )

        avg_style = (
            normalized_df.mean(
                axis=1
            )
        )

        std_style = (
            normalized_df.std(
                axis=1
            )
        )

        author_vec = (
            normalized_df[
                author
            ]
        )

        diff_vec = (
            author_vec
            - avg_style
        )

        val = [
            abs(x)
            for x
            in diff_vec
        ]


        if random_styles:

            sorted_indices = list(
                range(
                    df.shape[0]
                )
            )

            random.shuffle(
                sorted_indices
            )

        else:

            sorted_indices = sorted(
                range(
                    len(val)
                ),
                key=lambda i:
                    val[i],
                reverse=True,
            )


        direction = []
        magnitude = []
        styles_ls = []

        # StyleRemix allows only one writing-intent axis.
        type_selected = False


        for index in sorted_indices:

            style_name = (
                diff_vec.index[
                    index
                ]
            )


            # ------------------------------------------------
            # Writing intent:
            # choose a different intent category.
            # ------------------------------------------------

            if style_name in [
                "type_persuasive_conf",
                "type_narrative_conf",
                "type_expository_conf",
                "type_descriptive_conf",
            ]:

                if type_selected:
                    continue


                possible_types = [
                    x
                    for x in [
                        "type_persuasive_conf",
                        "type_narrative_conf",
                        "type_expository_conf",
                        "type_descriptive_conf",
                    ]
                    if x != style_name
                ]


                new_type = (
                    random.choice(
                        possible_types
                    )
                )


                styles_ls.append(
                    "old_"
                    + style_name
                    + "_new_"
                    + new_type
                )


                magnitude.append(
                    choose_weights(
                        val[index],

                        # IMPORTANT pandas compatibility fix.
                        std_style.iloc[
                            index
                        ],

                        selection_type=
                            magnitude_selection,
                    )
                )


                direction.append(
                    "new_type"
                )


                type_selected = True


            # ------------------------------------------------
            # Ordinary scalar style axes.
            # ------------------------------------------------

            else:

                # IMPORTANT pandas compatibility fix.
                if (
                    author_vec.iloc[
                        index
                    ]
                    >
                    avg_style.iloc[
                        index
                    ]
                ):

                    direction.append(
                        "higher"
                    )

                else:

                    direction.append(
                        "lower"
                    )


                magnitude.append(
                    choose_weights(
                        val[index],

                        # IMPORTANT pandas compatibility fix.
                        std_style.iloc[
                            index
                        ],

                        selection_type=
                            magnitude_selection,
                    )
                )


                styles_ls.append(
                    style_name
                )


            if (
                len(direction)
                == top_n
            ):
                break


        styles_to_change[
            author
        ] = {
            "styles":
                styles_ls,

            "direction":
                direction,

            "magnitude":
                magnitude,
        }


    return styles_to_change


# ------------------------------------------------------------
# Construct the exact StyleRemix style representation.
# ------------------------------------------------------------

style_df = avg.T[
    [
        "words_per_sent",
        "percent_function_words",
        "grade_level_average",

        "formal_conf",
        "sarcasm_conf",
        "voice_conf",

        "type_persuasive_conf",
        "type_narrative_conf",
        "type_expository_conf",
        "type_descriptive_conf",
    ]
].T


# Released utils.py seeds Python random to 0.
# Re-seed explicitly so this reproduction is deterministic.
random.seed(
    0
)


selected_all = (
    identify_styles_to_change_average_compat(
        style_df,

        top_n=TOP_K,

        # Paper's Adapter Merging heuristic.
        magnitude_selection="threshold",

        random_styles=False,
    )
)


selected = {
    author:
        selected_all[
            author
        ]

    for author
    in blog_data
}


print(
    "=== STYLE SELECTION ==="
)


for author, s in selected.items():

    print(
        "\n",
        author,
    )

    for style, direction, magnitude in zip(
        s["styles"],
        s["direction"],
        s["magnitude"],
    ):

        print(
            f"  {style:45s}"
            f" direction={direction:10s}"
            f" weight={magnitude:.3f}"
        )


with open(
    OUT
    / "selection.json",
    "w",
) as f:

    json.dump(
        selected,
        f,
        indent=2,
    )


print(
    "\nsaved:",
    OUT
    / "selection.json",
)

=== STYLE SELECTION ===

 blog_5_1
  old_type_persuasive_conf_new_type_expository_conf direction=new_type   weight=0.900
  formal_conf                                   direction=higher     weight=0.700
  grade_level_average                           direction=higher     weight=0.900

 blog_5_2
  old_type_persuasive_conf_new_type_expository_conf direction=new_type   weight=0.700
  words_per_sent                                direction=lower      weight=0.700
  formal_conf                                   direction=higher     weight=0.700

 blog_5_3
  formal_conf                                   direction=lower      weight=0.900
  sarcasm_conf                                  direction=higher     weight=0.900
  old_type_persuasive_conf_new_type_expository_conf direction=new_type   weight=0.700

 blog_5_4
  old_type_persuasive_conf_new_type_descriptive_conf direction=new_type   weight=0.700
  formal_conf                                   direction=higher     weight=0.700
  sarcasm_con

/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/nltk/downloader.py:1076: UserWarning: NLTK will not authorize the non-private download directory '/home/user/nltk_data': it (or an ancestor) is world- or group-writable, so another local user could plant files there. Choose a private location such as ~/nltk_data.
  for msg in self.incr_download(info_or_id, download_dir, force):
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/user/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!



## Load Llama-3 8B + only the released LoRAs needed for Blog

You need Hugging Face access to `meta-llama/Meta-Llama-3-8B`.


In [9]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print(
    "allocated:",
    round(torch.cuda.memory_allocated() / 1e9, 3),
    "GB"
)

print(
    "reserved:",
    round(torch.cuda.memory_reserved() / 1e9, 3),
    "GB"
)

print(
    "free / total:",
    tuple(
        round(x / 1e9, 3)
        for x in torch.cuda.mem_get_info()
    ),
    "GB"
)

allocated: 0.0 GB
reserved: 0.0 GB
free / total: (8.067, 8.187) GB


In [10]:
import subprocess

print(
    subprocess.run(
        [
            "nvidia-smi",
            "--query-compute-apps=pid,process_name,used_memory",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
    ).stdout
)

11187, /home/user/Downloads/TemporalContextResearch/.venv/bin/python, 94



In [11]:
!nvidia-smi

Thu Sep  3 12:31:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.84                 Driver Version: 595.84         CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 2000 Ada Gene...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   43C    P8              5W /   35W |     114MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
# # ============================================================
# # Wrap the already-loaded Llama base with the first LoRA,
# # then load the remaining required StyleRemix adapters.
# # ============================================================

# from peft import PeftModel
# import torch
# import gc

# first = sorted(needed)[0]

# print("first adapter:", first)

# model = PeftModel.from_pretrained(
#     base,
#     first,
#     adapter_name=repo_name[first],
#     is_trainable=False,
# )

# print(
#     "loaded 1 /",
#     len(needed),
# )


# for i, r in enumerate(
#     sorted(needed)[1:],
#     start=2,
# ):

#     print(
#         f"loading {i}/{len(needed)}:",
#         r,
#     )

#     model.load_adapter(
#         r,
#         adapter_name=repo_name[r],
#         is_trainable=False,
#     )

#     gc.collect()

#     if torch.cuda.is_available():
#         torch.cuda.empty_cache()

#         print(
#             "  allocated:",
#             round(
#                 torch.cuda.memory_allocated()
#                 / 1e9,
#                 3,
#             ),
#             "GB | reserved:",
#             round(
#                 torch.cuda.memory_reserved()
#                 / 1e9,
#                 3,
#             ),
#             "GB",
#         )


# model.eval()

# print("\nPASS: model + all required StyleRemix LoRAs loaded.")

In [13]:

from src.generate_lora_adapters import extract_lora_adapters

MODEL_ID = "meta-llama/Meta-Llama-3-8B"

author_repos = {}
needed = set()
for a in blog_data:
    repos = extract_lora_adapters(selected[a]["styles"], selected[a]["direction"])
    author_repos[a] = repos
    needed.update(repos)

print("required LoRAs:", len(needed))
for x in sorted(needed): print(" ", x)

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(
    MODEL_ID, add_bos_token=True, add_eos_token=False, padding_side="left"
)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

if USE_4BIT:
    qconf = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=qconf,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
else:
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto", low_cpu_mem_usage=True
    )


# ============================================================
# TEST ONE StyleRemix adapter
#
# - Stage giant checkpoint on CPU
# - Ignore the 128257-vs-128256 padding-token embeddings
# - Load the actual LoRA q_proj/v_proj deltas
# ============================================================

from peft import PeftModel
import torch
import gc

first = sorted(needed)[0]

print("testing adapter:")
print(first)

before_alloc = torch.cuda.memory_allocated()

model = PeftModel.from_pretrained(
    base,
    first,

    adapter_name="test_adapter",

    is_trainable=False,

    # Critical: do NOT materialize the 4.23 GB
    # checkpoint directly on CUDA.
    torch_device="cpu",

    # Normal state-dict copy is preferable here.
    low_cpu_mem_usage=False,

    autocast_adapter_dtype=False,

    # Skip the auto-saved 128257-row embedding/lm_head.
    # The actual Llama base has 128256 vocabulary rows.
    ignore_mismatched_sizes=True,
)

model.eval()

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

after_alloc = torch.cuda.memory_allocated()

print("\nPASS: adapter loaded")

print(
    "GPU allocated:",
    round(after_alloc / 1e9, 3),
    "GB",
)

print(
    "adapter persistent delta:",
    round(
        (after_alloc - before_alloc) / 1e9,
        3,
    ),
    "GB",
)

print(
    "LoRA parameter devices:",
    sorted({
        str(p.device)
        for name, p in model.named_parameters()
        if "lora_" in name
    }),
)

print(
    "LoRA parameter count:",
    sum(
        p.numel()
        for name, p in model.named_parameters()
        if "lora_" in name
    ),
)
# repo_name = {
#     r: f"a{i}"
#     for i, r in enumerate(
#         sorted(needed)
#     )
# }

# first = sorted(needed)[0]

# model = PeftModel.from_pretrained(
#     base,
#     first,
#     adapter_name=repo_name[first],
#     is_trainable=False,
# )

# for r in sorted(needed)[1:]:
#     model.load_adapter(
#         r,
#         adapter_name=repo_name[r],
#         is_trainable=False,
#     )

# model.eval()

# print("model ready")


required LoRAs: 10
  hallisky/lora-formality-formal-llama-3-8b
  hallisky/lora-formality-informal-llama-3-8b
  hallisky/lora-grade-elementary-llama-3-8b
  hallisky/lora-length-long-llama-3-8b
  hallisky/lora-sarcasm-less-llama-3-8b
  hallisky/lora-sarcasm-more-llama-3-8b
  hallisky/lora-type-descriptive-llama-3-8b
  hallisky/lora-type-expository-llama-3-8b
  hallisky/lora-type-narrative-llama-3-8b
  hallisky/lora-voice-passive-llama-3-8b


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████| 291/291 [00:10<00:00, 28.75it/s]


testing adapter:
hallisky/lora-formality-formal-llama-3-8b

PASS: adapter loaded
GPU allocated: 5.721 GB
adapter persistent delta: 0.014 GB
LoRA parameter devices: ['cuda:0']
LoRA parameter count: 6815744


/home/user/Downloads/TemporalContextResearch/.venv/lib/python3.12/site-packages/peft/utils/save_and_load.py:966: UserWarning: Some weights of PeftModelForCausalLM were not initialized from the model checkpoint and are being ignored because you passed `ignore_mismatched_sizes=True`: - base_model.model.lm_head.weight: found shape torch.Size([128257, 4096]) in the checkpoint and torch.Size([128256, 4096]) in the model instantiated
- base_model.model.model.embed_tokens.weight: found shape torch.Size([128257, 4096]) in the checkpoint and torch.Size([128256, 4096]) in the model instantiated.
  warnings.warn(msg)


In [14]:
# ============================================================
# Load the remaining required StyleRemix LoRAs safely
# ============================================================

import gc
import torch

repo_name = {
    r: f"a{i}"
    for i, r in enumerate(sorted(needed))
}

first = sorted(needed)[0]

# Rename conceptually: first repo is already loaded as test_adapter.
repo_name[first] = "test_adapter"

for i, repo in enumerate(
    sorted(needed)[1:],
    start=2,
):

    name = repo_name[repo]

    print(
        f"\nloading {i}/{len(needed)}:"
        f"\n  {repo}"
        f"\n  as {name}"
    )

    before = torch.cuda.memory_allocated()

    model.load_adapter(
        repo,
        adapter_name=name,
        is_trainable=False,

        # Stage the giant checkpoint on CPU.
        torch_device="cpu",

        # Avoid dtype promotion.
        autocast_adapter_dtype=False,

        # Skip the useless 128257-row saved embeddings.
        ignore_mismatched_sizes=True,
    )

    gc.collect()
    torch.cuda.empty_cache()

    after = torch.cuda.memory_allocated()

    print(
        "  persistent delta:",
        round(
            (after - before) / 1e9,
            3,
        ),
        "GB",
    )

    print(
        "  total allocated:",
        round(
            after / 1e9,
            3,
        ),
        "GB",
    )


model.eval()

print(
    "\nPASS: all required StyleRemix LoRAs loaded."
)

print(
    "final allocated:",
    round(
        torch.cuda.memory_allocated() / 1e9,
        3,
    ),
    "GB",
)


loading 2/10:
  hallisky/lora-formality-informal-llama-3-8b
  as a1
  persistent delta: 0.014 GB
  total allocated: 5.735 GB

loading 3/10:
  hallisky/lora-grade-elementary-llama-3-8b
  as a2
  persistent delta: 0.014 GB
  total allocated: 5.748 GB

loading 4/10:
  hallisky/lora-length-long-llama-3-8b
  as a3
  persistent delta: 0.014 GB
  total allocated: 5.762 GB

loading 5/10:
  hallisky/lora-sarcasm-less-llama-3-8b
  as a4
  persistent delta: 0.014 GB
  total allocated: 5.776 GB

loading 6/10:
  hallisky/lora-sarcasm-more-llama-3-8b
  as a5
  persistent delta: 0.014 GB
  total allocated: 5.789 GB

loading 7/10:
  hallisky/lora-type-descriptive-llama-3-8b
  as a6
  persistent delta: 0.014 GB
  total allocated: 5.803 GB

loading 8/10:
  hallisky/lora-type-expository-llama-3-8b
  as a7
  persistent delta: 0.014 GB
  total allocated: 5.817 GB

loading 9/10:
  hallisky/lora-type-narrative-llama-3-8b
  as a8
  persistent delta: 0.014 GB
  total allocated: 5.83 GB

loading 10/10:
  halli


### Build the merged per-author adapters

The next cell accepts the released selection dictionary's stored weight field. If the released schema differs, it raises instead of guessing.


In [15]:
def selected_weights(
    selection,
):

    return list(
        map(
            float,
            selection[
                "magnitude"
            ],
        )
    )

combo = {}
for a in blog_data:
    weights = selected_weights(selected[a])
    names = [repo_name[r] for r in author_repos[a]]
    assert len(weights)==len(names)==TOP_K, (a, weights, names)
    cname = "sr_" + a
    print(a, list(zip(names,weights)))
    model.add_weighted_adapter(
        names, weights=weights, adapter_name=cname, combination_type="cat"
    )
    combo[a]=cname

print("PASS")


blog_5_1 [('a7', 0.9), ('a1', 0.7), ('a2', 0.9)]
blog_5_2 [('a7', 0.7), ('a3', 0.7), ('a1', 0.7)]
blog_5_3 [('test_adapter', 0.9), ('a4', 0.9), ('a7', 0.7)]
blog_5_4 [('a6', 0.7), ('a1', 0.7), ('a5', 0.7)]
blog_5_5 [('a8', 0.7), ('test_adapter', 0.7), ('a9', 0.9)]
PASS



## Generate with per-example checkpointing

Paper generation uses `top_p=.95` and up to 512 new tokens. Every completed sample is saved immediately.


In [17]:
# ============================================================
# Restore StyleRemix generation checkpoint after kernel restart
# ============================================================

from pathlib import Path
import pandas as pd

ROOT = Path(
    ".."
)

OUT = (
    ROOT
    / "results"
    / "causal"
    / "C02a_styleremix_reproduction"
)

GEN_PATH = (
    OUT
    / "styleremix_blog_generations.parquet"
)

print("checkpoint:", GEN_PATH)
print("exists:", GEN_PATH.exists())

if GEN_PATH.exists():

    gen = pd.read_parquet(
        GEN_PATH
    )

    done = set(
        zip(
            gen["author"].astype(str),
            gen["index"].astype(int),
        )
    )

    print(
        "saved generations:",
        len(gen),
        "/ 1000"
    )

    print("\nby author:")
    display(
        gen.groupby("author")
        .size()
        .rename("saved")
        .to_frame()
    )

else:

    raise FileNotFoundError(
        f"Checkpoint not found:\n{GEN_PATH}"
    )

checkpoint: ../results/causal/C02a_styleremix_reproduction/styleremix_blog_generations.parquet
exists: True
saved generations: 689 / 1000

by author:


,saved
author,
blog_5_1,200
blog_5_2,200
blog_5_3,200
blog_5_4,89


In [19]:
# ============================================================
# Restore StyleRemix rewrite function after kernel restart
# ============================================================

import gc
import torch

@torch.inference_mode()
def rewrite_one(
    author,
    text,
):
    # Select this author's merged StyleRemix adapter.
    model.set_adapter(
        combo[author]
    )

    prompt = (
        f"### Original: {text}\n"
        f" ### Rewrite:"
    )

    enc = tok(
        prompt,
        return_tensors="pt",
        max_length=256,
        truncation=True,
    )

    enc = {
        k: v.to(model.device)
        for k, v in enc.items()
    }

    input_len = (
        enc["input_ids"]
        .shape[1]
    )

    y = model.generate(
        **enc,
        max_new_tokens=512,
        do_sample=True,
        top_p=0.95,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )

    out = tok.decode(
        y[0, input_len:],
        skip_special_tokens=True,
    ).strip()

    # Match StyleRemix's released output cleaning.
    if out.startswith("###"):
        out = out[3:].strip()

    out = (
        out.split("###")[0]
        .strip()
    )

    del enc, y

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


print("PASS: rewrite_one restored.")

PASS: rewrite_one restored.


In [20]:
from tqdm.auto import tqdm
import warnings
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore")

expected = sum(
    len(x)
    for x in blog_data.values()
)

# Load existing checkpoint if present.
if GEN_PATH.exists():
    gen = pd.read_parquet(
        GEN_PATH
    )
else:
    gen = pd.DataFrame(
        columns=[
            "author",
            "index",
            "original",
            "rewrite",
            "seconds",
        ]
    )

done = set(
    zip(
        gen["author"].astype(str),
        gen["index"].astype(int),
    )
)

jobs = [
    (
        author,
        i,
        original,
    )
    for author, texts
    in blog_data.items()
    for i, original
    in enumerate(texts)
    if (
        author,
        i,
    ) not in done
]

print(
    f"already saved: "
    f"{len(done)}/{expected}"
)

pbar = tqdm(
    jobs,
    total=len(jobs),
    desc="StyleRemix",
    unit="text",
    dynamic_ncols=True,
)

for author, i, original in pbar:

    t0 = time.time()

    rewrite = rewrite_one(
        author,
        original,
    )

    seconds = (
        time.time()
        - t0
    )

    row = pd.DataFrame([{
        "author": author,
        "index": i,
        "original": original,
        "rewrite": rewrite,
        "seconds": seconds,
    }])

    gen = pd.concat(
        [gen, row],
        ignore_index=True,
    )

    # Save IMMEDIATELY after each completed sample.
    gen.to_parquet(
        GEN_PATH,
        index=False,
    )

    done.add(
        (
            author,
            i,
        )
    )

    pbar.set_postfix(
        {
            "saved":
                f"{len(done)}/{expected}",
            "last":
                f"{seconds:.2f}s",
        },
        refresh=False,
    )

print(
    "\nPASS:",
    len(done),
    "/",
    expected,
    "saved"
)

print(
    "checkpoint:",
    GEN_PATH,
)

already saved: 689/1000


StyleRemix: 100%|█████████████████████████████████████████████████████| 311/311 [18:52<00:00,  3.64s/text, saved=1000/1000, last=3.43s]


PASS: 1000 / 1000 saved
checkpoint: ../results/causal/C02a_styleremix_reproduction/styleremix_blog_generations.parquet


In [21]:

# Free 8B generator before evaluation.
del model, base
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print("generator freed")


generator freed



## Evaluate the released paper metrics


In [22]:

gen=pd.read_parquet(GEN_PATH)
assert len(gen)==expected

LABELS = {
    "blog_5_1":"blog5546","blog_5_2":"blog11518","blog_5_3":"blog25872",
    "blog_5_4":"blog30102","blog_5_5":"blog30407"
}

auth = pipeline(
    "text-classification",
    model="hallisky/blog-classifier-roberta-large",
    tokenizer="roberta-large",
    top_k=None, function_to_apply="sigmoid", device=0, truncation=True
)

def pred(text):
    out=auth(text)[0]
    return max(out,key=lambda z:z["score"])["label"]

gen["truth"]=gen.author.map(LABELS)
gen["pred_orig"]=[pred(x) for x in gen.original]
gen["pred_obf"]=[pred(x) for x in gen.rewrite]

acc_orig=np.mean(gen.pred_orig==gen.truth)
acc_obf=np.mean(gen.pred_obf==gen.truth)
drop=(acc_orig-acc_obf)/acc_orig

print("original acc:",acc_orig)
print("obfuscated acc:",acc_obf)
print("drop rate:",drop)


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 393/393 [00:00<00:00, 11049.78it/s]


original acc: 0.903
obfuscated acc: 0.543
drop rate: 0.3986710963455149


In [23]:

del auth
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

cola_tok=AutoTokenizer.from_pretrained("textattack/roberta-base-CoLA")
cola=AutoModelForSequenceClassification.from_pretrained(
    "textattack/roberta-base-CoLA"
).to("cuda").eval()

@torch.inference_mode()
def cola_piece(text):
    toks=cola_tok.tokenize(text)
    ids=torch.tensor([cola_tok.convert_tokens_to_ids(toks)],device="cuda")
    return cola(ids).logits.softmax(-1)[0,1].item()

def grammar_score(text):
    if not text: return np.nan
    if len(text)>300:
        ng=int(np.ceil(len(text)/200))
        vals=[]
        for n in range(ng):
            lo=n*int(len(text)/ng); hi=(n+1)*int(len(text)/ng)
            vals.append(cola_piece(text[lo:hi]))
        return float(np.mean(vals))
    return cola_piece(text)

gen["grammar"]=[grammar_score(x) for x in gen.rewrite]
grammar=float(gen.grammar.mean())
print("grammar:",grammar)


Loading weights: 100%|████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 45293.89it/s]


grammar: 0.6339208055008025


In [24]:

del cola
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

sim=SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",device="cuda")
a=sim.encode(gen.original.tolist(),convert_to_tensor=True,show_progress_bar=True)
b=sim.encode(gen.rewrite.tolist(),convert_to_tensor=True,show_progress_bar=True)
gen["content_similarity"]=torch.nn.functional.cosine_similarity(a,b).cpu().numpy()
content=float(gen.content_similarity.mean())
overall=drop*grammar*content

summary=pd.DataFrame([
    ["drop_rate",drop,.410],
    ["grammar",grammar,.649],
    ["content",content,.737],
    ["overall",overall,.196],
],columns=["metric","ours","paper"])
summary["abs_error"]=(summary.ours-summary.paper).abs()
summary["ours_x100"]=100*summary.ours
summary["paper_x100"]=100*summary.paper
display(summary)

summary.to_csv(OUT/"summary.csv",index=False)
gen.to_parquet(OUT/"evaluated.parquet",index=False)

with open(OUT/"metadata.json","w") as f:
    json.dump({
        "commit":commit,"top_k":TOP_K,"n":len(gen),"seed":SEED,
        "base_model":MODEL_ID,
        "precision":"4bit_nf4" if USE_4BIT else "nonquantized"
    },f,indent=2)

print("saved:",OUT)


Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 32/32 [00:00<00:00, 44.32it/s]


,metric,ours,paper,abs_error,ours_x100,paper_x100
0,drop_rate,0.398671,0.410,0.011329,39.867110,41.0
1,grammar,0.633921,0.649,0.015079,63.392081,64.9
2,content,0.729350,0.737,0.007650,72.935021,73.7
3,overall,0.184326,0.196,0.011674,18.432569,19.6


saved: ../results/causal/C02a_styleremix_reproduction


In [25]:

# Manual sanity audit
for _,r in gen.sample(min(5,len(gen)),random_state=SEED).iterrows():
    print("="*100)
    print(r.author, "| grammar",round(r.grammar,3),
          "| content",round(r.content_similarity,3))
    print("\nORIGINAL\n",r.original)
    print("\nSTYLEREMIX\n",r.rewrite)
    print()


blog_5_3 | grammar 0.32 | content 0.639

ORIGINAL
 He kept wondering how we write stuff if we don't have an alphabet lol.  But yeah in the end I think he finally understood we use CHARACTERS and not every language is based on ENGLISH RULES.  Bio class Melissa's group went today for seminars.  Our group really has to start working on ours. 

STYLEREMIX
 He consistently pondered the notion of how we communicate without the utilization of an alphabet. Additionally, his perspective evolved, acknowledging that various communication systems employ different alphabetic systems, distinct from those based on English conventions. Today, the group assigned to Melissa in the biology class attended seminars. It becomes imperative that our group begins to actively prepare our presentation.

blog_5_4 | grammar 0.645 | content 0.539

ORIGINAL
 How do you manage to deal with being an outcast?  Is being crazy really an answer?  If I went out of my mind and did whatever I wanted to do, would I be granted

In [26]:
gen.to_parquet(
    OUT / "styleremix_blog_authorship_eval.parquet",
    index=False,
)

In [27]:
# ============================================================
# C02a — verify StyleRemix reproduction artifacts are saved
# ============================================================

from pathlib import Path
import json
import pandas as pd

ROOT = Path(
    ".."
)

OUT = (
    ROOT
    / "results"
    / "causal"
    / "C02a_styleremix_reproduction"
)

EXPECTED_N = 1000

files = {
    "raw_generations":
        OUT / "styleremix_blog_generations.parquet",

    "authorship_eval":
        OUT / "styleremix_blog_authorship_eval.parquet",

    "evaluated":
        OUT / "evaluated.parquet",

    "summary":
        OUT / "summary.csv",

    "metadata":
        OUT / "metadata.json",

    "selection":
        OUT / "selection.json",
}


print("=== FILE EXISTENCE ===")

for name, path in files.items():
    print(
        f"{name:18s}",
        "OK" if path.exists() else "MISSING",
        "|",
        path,
    )


# ------------------------------------------------------------
# 1. Raw generations — MOST IMPORTANT FILE
# ------------------------------------------------------------

assert files["raw_generations"].exists(), (
    "Missing raw generations checkpoint!"
)

raw = pd.read_parquet(
    files["raw_generations"]
)

required_raw_cols = {
    "author",
    "index",
    "original",
    "rewrite",
    "seconds",
}

assert required_raw_cols.issubset(raw.columns), (
    f"Raw generations missing columns: "
    f"{required_raw_cols - set(raw.columns)}"
)

assert len(raw) == EXPECTED_N, (
    f"Expected {EXPECTED_N} raw generations, got {len(raw)}"
)

assert not raw.duplicated(
    ["author", "index"]
).any(), "Duplicate author/index rows in raw generations!"

assert raw["rewrite"].notna().all(), (
    "Some rewrites are NaN"
)

assert (
    raw["rewrite"]
    .astype(str)
    .str.strip()
    .ne("")
    .all()
), "Some rewrites are blank"


print("\n=== RAW GENERATIONS ===")
print("rows:", len(raw))
print(
    "unique author/index:",
    raw[["author", "index"]]
    .drop_duplicates()
    .shape[0],
)

display(
    raw.groupby("author")
    .size()
    .rename("n")
    .to_frame()
)


# ------------------------------------------------------------
# 2. Evaluated output
# ------------------------------------------------------------

if files["evaluated"].exists():

    evaluated = pd.read_parquet(
        files["evaluated"]
    )

    print("\n=== EVALUATED ===")
    print("rows:", len(evaluated))
    print("columns:")
    print(list(evaluated.columns))

    assert len(evaluated) == EXPECTED_N

    expected_eval_cols = {
        "author",
        "index",
        "original",
        "rewrite",
        "grammar",
        "content_similarity",
    }

    missing_eval = (
        expected_eval_cols
        - set(evaluated.columns)
    )

    assert not missing_eval, (
        f"evaluated.parquet missing: {missing_eval}"
    )


# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

if files["summary"].exists():

    summary = pd.read_csv(
        files["summary"]
    )

    print("\n=== SUMMARY ===")
    display(summary)

    expected_metrics = {
        "drop_rate",
        "grammar",
        "content",
        "overall",
    }

    assert expected_metrics.issubset(
        set(summary["metric"])
    )


# ------------------------------------------------------------
# 4. Metadata
# ------------------------------------------------------------

if files["metadata"].exists():

    with open(
        files["metadata"],
        "r",
    ) as f:
        metadata = json.load(f)

    print("\n=== METADATA ===")
    print(
        json.dumps(
            metadata,
            indent=2,
        )
    )


# ------------------------------------------------------------
# 5. Style selection
# ------------------------------------------------------------

if files["selection"].exists():

    with open(
        files["selection"],
        "r",
    ) as f:
        selection = json.load(f)

    print("\n=== STYLE SELECTION ===")
    print("authors:", len(selection))

    for author, s in selection.items():
        print(author, "->", s)


# ------------------------------------------------------------
# Final verdict
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PASS: C02a StyleRemix artifacts are safely saved.")
print()
print("MOST IMPORTANT ASSET:")
print(files["raw_generations"])
print("=" * 80)

=== FILE EXISTENCE ===
raw_generations    OK | ../results/causal/C02a_styleremix_reproduction/styleremix_blog_generations.parquet
authorship_eval    OK | ../results/causal/C02a_styleremix_reproduction/styleremix_blog_authorship_eval.parquet
evaluated          OK | ../results/causal/C02a_styleremix_reproduction/evaluated.parquet
summary            OK | ../results/causal/C02a_styleremix_reproduction/summary.csv
metadata           OK | ../results/causal/C02a_styleremix_reproduction/metadata.json
selection          OK | ../results/causal/C02a_styleremix_reproduction/selection.json

=== RAW GENERATIONS ===
rows: 1000
unique author/index: 1000


,n
author,
blog_5_1,200
blog_5_2,200
blog_5_3,200
blog_5_4,200
blog_5_5,200



=== EVALUATED ===
rows: 1000
columns:
['author', 'index', 'original', 'rewrite', 'seconds', 'truth', 'pred_orig', 'pred_obf', 'grammar', 'content_similarity']

=== SUMMARY ===


,metric,ours,paper,abs_error,ours_x100,paper_x100
0,drop_rate,0.398671,0.410,0.011329,39.867110,41.0
1,grammar,0.633921,0.649,0.015079,63.392081,64.9
2,content,0.729350,0.737,0.007650,72.935021,73.7
3,overall,0.184326,0.196,0.011674,18.432569,19.6



=== METADATA ===
{
  "commit": "c999002c4a5e858ac452794b20b896d06137a13c",
  "top_k": 3,
  "n": 1000,
  "seed": 42,
  "base_model": "meta-llama/Meta-Llama-3-8B",
  "precision": "4bit_nf4"
}

=== STYLE SELECTION ===
authors: 5
blog_5_1 -> {'styles': ['old_type_persuasive_conf_new_type_expository_conf', 'formal_conf', 'grade_level_average'], 'direction': ['new_type', 'higher', 'higher'], 'magnitude': [0.9, 0.7, 0.9]}
blog_5_2 -> {'styles': ['old_type_persuasive_conf_new_type_expository_conf', 'words_per_sent', 'formal_conf'], 'direction': ['new_type', 'lower', 'higher'], 'magnitude': [0.7, 0.7, 0.7]}
blog_5_3 -> {'styles': ['formal_conf', 'sarcasm_conf', 'old_type_persuasive_conf_new_type_expository_conf'], 'direction': ['lower', 'higher', 'new_type'], 'magnitude': [0.9, 0.9, 0.7]}
blog_5_4 -> {'styles': ['old_type_persuasive_conf_new_type_descriptive_conf', 'formal_conf', 'sarcasm_conf'], 'direction': ['new_type', 'higher', 'lower'], 'magnitude': [0.7, 0.7, 0.7]}
blog_5_5 -> {'styles':